# 🛡️ Notebook 4: Timeouts & Graceful Degradation

Two resilience patterns that pair with everything else in this lab:

1. **Timeouts** — the single most important resilience pattern. Without them, a slow downstream becomes an infinite wait, and all the retry/breaker/bulkhead tricks in the world can't save you.
2. **Graceful degradation** — when something *does* fail, serve a *degraded* answer instead of a broken page. 'Slightly worse product' beats '500 Internal Server Error'.

> Netflix motto: *"Better to serve you a stale recommendation than no page at all."*

## 🛠️ Setup

```bash
cd 04-patterns/resilience
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## ⏱️ Part 1 — Timeouts

### 🟥 BAD: no timeout

A downstream that never responds will hold our thread forever. One such call is survivable; a stream of them exhausts the thread pool and the whole service goes down. This is how partial outages become total outages.

In [1]:
import time, threading

def hanging_call():
    # simulates a downstream that stops responding
    time.sleep(30)
    return 'ok'

# We'll NOT actually run this — it'd hang the notebook for 30s.
# But imagine a web server with 100 threads, and 100 of these in flight: every
# incoming request (even healthy ones) now queues forever.
print('skipping — this would block 30s')


skipping — this would block 30s


### 🟩 GOOD: a hard timeout

Real HTTP clients (`requests`, `httpx`, `aiohttp`) have a `timeout=` argument — *always set it*. Below we show the general idea with a worker thread and `Event.wait()`:

In [2]:
class TimeoutError_(Exception): pass

def call_with_timeout(fn, timeout):
    """Run fn() in a background thread; raise if it doesn't finish in `timeout` seconds."""
    result, error, done = [], [], threading.Event()

    def worker():
        try: result.append(fn())
        except Exception as e: error.append(e)
        finally: done.set()

    threading.Thread(target=worker, daemon=True).start()
    if not done.wait(timeout):
        raise TimeoutError_(f'call exceeded {timeout}s')
    if error: raise error[0]
    return result[0]

t0 = time.perf_counter()
try:
    call_with_timeout(hanging_call, timeout=0.3)
except TimeoutError_ as e:
    print(f'timed out after {time.perf_counter()-t0:.2f}s:', e)


timed out after 0.37s: call exceeded 0.3s


### 🌍 Real-world rules

- **Always set timeouts** on every network call. `requests.get(url)` with no `timeout=` is a production incident waiting to happen.
- **Pick sensible values** — usually a few multiples of the 99th-percentile latency, *not* a huge 'safe' default like 60 s.
- **Timeouts should shrink as you go deeper** — if the user-facing API has a 2 s budget, the DB call within it should have maybe 500 ms. Otherwise the outer timeout fires while the inner call is still waiting.
- **Caller timeout < server timeout** — if the caller gives up first, the server is still working on a dead request. Coordinate them.

In async Python, use [`asyncio.timeout(...)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.timeout):
```python
async with asyncio.timeout(0.3):
    await call_downstream()
```

## 💔 Part 2 — Graceful degradation

When an optional dependency fails, don't bubble the error to the user. Return something *good enough*:

| Feature | If dependency fails, serve… |
|---|---|
| Personalized recommendations | a generic 'popular products' list |
| User avatar | a default silhouette image |
| Friend activity feed | an empty list, not a crashed page |
| Currency conversion | last-known exchange rate from cache |
| Search with typo-correction | raw search results without correction |

### 🟥 BAD: one failure breaks the page

In [3]:
def recommendations_service():
    raise IOError('recommendations ML service down')

def product_page_bad(user):
    recs = recommendations_service()   # <- raises
    return {'user': user, 'recommendations': recs}

try:
    page = product_page_bad('alice')
except IOError as e:
    print('page rendering FAILED:', e, '— user sees a 500')


page rendering FAILED: recommendations ML service down — user sees a 500


### 🟩 GOOD: fallback to a default

Catch the failure at the edge of the feature and substitute a sensible default. The user still gets a product page — just without personalization.

In [4]:
POPULAR_FALLBACK = ['socks', 't-shirt', 'mug']   # safe, non-personalized default

def get_recommendations(user):
    try:
        return recommendations_service()
    except Exception as e:
        print(f'  ⚠ recommendations failed ({e}); serving popular-items fallback')
        return POPULAR_FALLBACK

def product_page_good(user):
    return {'user': user, 'recommendations': get_recommendations(user)}

print(product_page_good('alice'))


  ⚠ recommendations failed (recommendations ML service down); serving popular-items fallback
{'user': 'alice', 'recommendations': ['socks', 't-shirt', 'mug']}


### 🧊 Even better: stale cache as fallback

Serve the *last good answer* instead of a generic default — users barely notice.

In [5]:
_cache = {}   # user -> last good recommendations

def get_recs_with_stale_cache(user):
    try:
        fresh = recommendations_service()
        _cache[user] = fresh   # refresh the cache on success
        return fresh
    except Exception:
        if user in _cache:
            print('  ℹ️ serving stale cache (recommendations service is down)')
            return _cache[user]
        print('  ⚠ no cache — falling back to popular items')
        return POPULAR_FALLBACK

# Pretend we had a good response yesterday:
_cache['alice'] = ['running shoes', 'water bottle', 'headband']
print(get_recs_with_stale_cache('alice'))
print(get_recs_with_stale_cache('bob'))


  ℹ️ serving stale cache (recommendations service is down)
['running shoes', 'water bottle', 'headband']
  ⚠ no cache — falling back to popular items
['socks', 't-shirt', 'mug']


## 🚦 Part 3 — Feature flags (kill switches)

Sometimes graceful degradation has to happen *before* the call even goes out. A **feature flag** (aka *kill switch*) lets on-call operators turn off an expensive feature in seconds without a deploy:

- 'Disable recommendations panel' — service-under-load relief valve.
- 'Disable image thumbnails' — skip the thumbnailer during the outage.
- 'Read-only mode' — turn off writes when the primary DB is in trouble.

Common libraries: [Unleash](https://www.getunleash.io/), [LaunchDarkly](https://launchdarkly.com/), or a simple config table in your database.

In [6]:
FEATURE_FLAGS = {
    'show_recommendations': True,   # on-call flips this to False during an outage
    'show_avatars': True,
}

def product_page_with_flags(user):
    page = {'user': user}
    if FEATURE_FLAGS['show_recommendations']:
        page['recommendations'] = get_recs_with_stale_cache(user)
    return page

print('feature ON :', product_page_with_flags('alice'))

# On-call disables the feature
FEATURE_FLAGS['show_recommendations'] = False
print('feature OFF:', product_page_with_flags('alice'))


  ℹ️ serving stale cache (recommendations service is down)
feature ON : {'user': 'alice', 'recommendations': ['running shoes', 'water bottle', 'headband']}
feature OFF: {'user': 'alice'}


## 🧠 Putting it all together

The patterns in this lab stack — each layer handles a *different* failure mode:

```
  user request
      |
      v
  feature flag        <- (cheap) turn off non-essential features under load
      |
      v
  bulkhead            <- limit concurrent calls per dependency
      |
      v
  circuit breaker     <- short-circuit if dependency is dead
      |
      v
  retry w/ jitter     <- survive transient blips
      |
      v
  timeout             <- never wait forever
      |
      v
  downstream service
```

…and on failure, **graceful degradation** / fallback gives the user a still-useful page.

| Pattern | Problem it solves |
|---|---|
| Timeout | a slow call that never returns |
| Retry + jitter | transient network blips |
| Circuit breaker | a sustained outage |
| Bulkhead | blast radius — one slow dep taking down unrelated ones |
| Graceful degradation / fallback | still serve *something* when things fail |
| Feature flag / kill switch | operator-controlled load shedding |
